# Ukázková úloha z PDF — rozbor

Oficiální zadání okruhu 2 (APR-I-II-3okruhy.pdf). Tohle je jediná úloha, o které víš jistě,
že je reprezentativní. Projdi ji celou, na časovku 60 minut.

**Postup:** nejdřív si přečti zadání, zavři tenhle notebook a zkus to sám.
Až potom se dívej na řešení dole.

## Zadání

> V následující funkci nalezněte syntaktické i sémantické chyby a opravte je.

Program modifikujte/rozšiřte o následující funkčnost:
- inkrementován bude prvek s lichým indexem (druhý, čtvrtý, šestý …)
- místo modifikace bude vrácen nový seznam
- kontrola, zda jsou prvky skutečně čísla, pokud nikoliv vyhození výjimky

**Výstup:** opravený program + výsledky ladění (co funguje a co nikoliv a proč?)

### Původní kód — spusť ho NEJDŘÍV tak, jak je

In [ ]:
# kód má ke každé položce modifikovatelné sekvence s celočíselnými prvky
# přičíst 1 je-li položka sudá

data = range(1, 10)

print(data)

for item in data:
    if item % 2 == 1:
        item += 1

print(data)

Co jsi viděl? Dvakrát `range(1, 10)` a **žádnou chybovou hlášku**.

To je klíčové pozorování: kód doběhne, ale nedělá nic z toho, co slibuje komentář.
Kdybys ho jen četl a nespustil, mohl bys uvěřit, že „nějak funguje".

---
## Tvoje řešení

Piš sem. Na řešení se dívej až potom.

In [ ]:
# 1) Oprava

In [ ]:
# 2) Rozšíření

In [ ]:
# 3) Testy

---
---
# ŘEŠENÍ — nedívej se, dokud nemáš svoje

## Nalezené chyby

| # | Řádek | Typ | Popis | Oprava |
|---|-------|-----|-------|--------|
| 1 | 4 | sémantická | `range` **není** modifikovatelná sekvence, ale líný generátor | `list(range(1, 10))` |
| 2 | 6 | sémantická | `print(data)` vypíše `range(1, 10)`, ne prvky | důsledek chyby 1 |
| 3 | 9 | sémantická | `item += 1` mění lokální proměnnou, ne prvek seznamu | zápis přes index `data[i]` |
| 4 | 8 | sémantická | `% 2 == 1` je liché, komentář říká **sudé** | `% 2 == 0` |

Ani jedna chyba není syntaktická — proto se kód spustí. Sémantické chyby najdeš **jen porovnáním
kódu s komentářem/docstringem**, nikdy ne spuštěním.

### Opravená verze — doslova podle komentáře

In [ ]:
data = list(range(1, 10))       # [1, 2, ..., 9]
print(data)

for i in range(len(data)):
    if data[i] % 2 == 0:        # sudá HODNOTA, podle komentáře
        data[i] += 1            # zápis přes index — tohle už mění seznam

print(data)                     # [1, 3, 3, 5, 5, 7, 7, 9, 9]

Ověření v hlavě: sudé hodnoty jsou 2, 4, 6, 8 → z nich bude 3, 5, 7, 9. Liché zůstanou.

Ekvivalentní zápis přes `enumerate` (čitelnější, u zkoušky ho preferuj):

In [ ]:
data = list(range(1, 10))
for i, hodnota in enumerate(data):
    if hodnota % 2 == 0:
        data[i] += 1
print(data)

### Rozšíření — všechny tři body ze zadání najednou

In [ ]:
def inkrementuj_liche_indexy(data: list) -> list:
    """
    Vrací NOVÝ seznam, kde jsou o 1 zvýšeny prvky na lichém indexu
    (tedy druhý, čtvrtý, šestý … prvek v pořadí).

    Vyhazuje TypeError, není-li vstup seznam.
    Vyhazuje ValueError, není-li některý prvek číslo.
    """
    if not isinstance(data, list):
        raise TypeError(f"očekávám seznam, dostal jsem {type(data).__name__}")

    # kontrola PŘEDEM, ne za pochodu — funkce buď uspěje celá, nebo neudělá nic
    for i, x in enumerate(data):
        if isinstance(x, bool) or not isinstance(x, (int, float)):
            raise ValueError(f"prvek na indexu {i} není číslo: {x!r}")

    return [x + 1 if i % 2 == 1 else x for i, x in enumerate(data)]


print(inkrementuj_liche_indexy([10, 20, 30, 40]))   # [10, 21, 30, 41] — indexy 1 a 3

**Proč `enumerate` a ne `data[1::2]`:** slicing vybere správné prvky, ale ztratí jejich pozice
a nešel by z něj složit celý seznam v původním pořadí.

**Proč kontrola typů v samostatném cyklu:** aby funkce buď uspěla celá, nebo neudělala nic.
U funkce vracející nový seznam je to jedno, u modifikace na místě zásadní — **a přesně na tohle
se u obhajoby ptají**.

**Proč `isinstance(x, bool)` zvlášť:** `bool` v Pythonu dědí z `int`, takže `isinstance(True, int)`
je `True`. Bez té kontroly by `[True, False]` prošlo jako „seznam čísel".

### Testy — včetně hraničních případů

In [ ]:
testy = [
    ([10, 20, 30, 40], [10, 21, 30, 41]),   # běžný případ
    ([], []),                                # prázdný seznam
    ([5], [5]),                              # jeden prvek — žádný lichý index
    ([1, 2], [1, 3]),                        # dva prvky
    ([1.5, 2.5], [1.5, 3.5]),                # floaty
]

for vstup, ocekavano in testy:
    vysledek = inkrementuj_liche_indexy(vstup)
    stav = "OK " if vysledek == ocekavano else "CHYBA"
    print(f"{stav} {vstup} -> {vysledek}  (čekáno {ocekavano})")

# vstup se nemodifikoval?
puvodni = [1, 2, 3]
inkrementuj_liche_indexy(puvodni)
print(f"vstup beze změny: {puvodni == [1, 2, 3]}")

In [ ]:
# chybové stavy
for spatny in [[1, "a"], [1, None], [True, False], "nejsem seznam"]:
    try:
        inkrementuj_liche_indexy(spatny)
        print(f"{spatny!r}: PROŠLO (nemělo!)")
    except (ValueError, TypeError) as e:
        print(f"{spatny!r}: {type(e).__name__}: {e}")

### Varianta na místě — umět obě a umět vysvětlit rozdíl

In [ ]:
def inkrementuj_na_miste(data: list) -> None:
    """Modifikuje seznam PŘÍMO. Nic nevrací (proto -> None)."""
    for i in range(1, len(data), 2):    # 1, 3, 5, … rovnou jen liché indexy
        data[i] += 1


s = [10, 20, 30, 40]
inkrementuj_na_miste(s)
print(s)   # [10, 21, 30, 41] — změnil se PŮVODNÍ seznam

| | Nový seznam | Modifikace na místě |
|---|---|---|
| vrací | nový `list` | `None` |
| vstup volajícího | beze změny | **změněn** (vedlejší efekt) |
| paměť | $O(n)$ navíc | $O(1)$ |
| kdy použít | výchozí volba, čistá funkce | velká data, kde kopie nevejde |

Časová složitost je v obou případech $O(n)$ — projdeš seznam jednou.

---
## Výsledky ladění

*(Tohle je požadovaný výstup ze zadání. U zkoušky vyplň analogicky.)*

### Co funguje
- **Oprava:** pro `[1..9]` vrací `[1,3,3,5,5,7,7,9,9]` — ověřeno ručním výpočtem (sudé 2,4,6,8 → 3,5,7,9).
- **Rozšíření:** `[10,20,30,40]` → `[10,21,30,41]`, tj. indexy 1 a 3.
- **Nový seznam:** ověřeno, že vstupní seznam zůstává beze změny.
- **Kontrola typů:** `[1,"a"]` vyhodí `ValueError` s indexem chybného prvku.
- **Hraniční případy:** prázdný seznam a jednoprvkový vrací kopii vstupu (žádný lichý index).

### Co nefunguje / vědomá omezení
- `bool` je v Pythonu podtyp `int`; ošetřeno zvlášť, jinak by `[True, False]` prošlo jako čísla.
- Nekontroluje se vnořený seznam — `[[1,2]]` vyhodí `ValueError`, což je zamýšlené chování.
- `complex` je odmítnut, přestože je to číslo. Zadání mluví o celočíselných prvcích, takže OK.

### Jak jsem testoval
- Krátké seznamy s ručně dopočítaným výsledkem, ne generovaná data.
- Hraniční případy: prázdný, jednoprvkový, dvouprvkový.
- Chybové stavy: nečíselný prvek, `None`, `bool`, nesprávný typ vstupu.